In [ ]:
# !pip install Sastrawi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install optuna
!pip install requests -q

### Sken 1 : indoBERT, GridSearch, Class Weight

In [ ]:
# Cell 1: Imports and Configuration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_recall_fscore_support, roc_curve, auc
)
from sklearn.preprocessing import LabelEncoder, label_binarize
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
import re
import warnings
import pickle
import os
from datetime import datetime
import json
from collections import Counter
import gc
from sklearn.utils.class_weight import compute_class_weight
import requests
import json
warnings.filterwarnings('ignore')

class Config:
    MODEL_NAME = "indobenchmark/indobert-large-p2"
    MAX_LENGTH = 128
    BATCH_SIZE = 32
    EVAL_BATCH_SIZE = 32
    NUM_EPOCHS = 10
    LEARNING_RATE = 4e-5
    WARMUP_RATIO = 0.1
    WEIGHT_DECAY = 0.01
    TEST_SIZE = 0.2
    VAL_SIZE = 0.1
    RANDOM_STATE = 42
    DATA_PATH = '/content/drive/MyDrive/Telkom (1)/Proposal/PRDECT-ID Dataset.csv'
    BASE_OUTPUT_PATH = '/content/drive/MyDrive/Telkom (1)/Proposal/hasil_sken1'
    MODEL_SAVE_PATH = f"{BASE_OUTPUT_PATH}/fine_tuned_indobert_emotion"
    RESULTS_PATH = f"{BASE_OUTPUT_PATH}/results"
    LOGS_PATH = f"{BASE_OUTPUT_PATH}/logs"

In [ ]:
# Cell 2: Environment Setup
def setup_environment():
    print("="*60)
    print("SETUP ENVIRONMENT")
    print("="*60)
    plt.style.use('default')
    sns.set_palette("husl")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"CUDA Version: {torch.version.cuda}")
        print(f"Available GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    for path in [Config.RESULTS_PATH, Config.LOGS_PATH, 'models']:
        os.makedirs(path, exist_ok=True)
    return device

device = setup_environment()

In [ ]:
# Cell 3: Data Loading, Filtering, and Exploration (LENGKAP DENGAN PERBANDINGAN KATEGORI)

def visualize_data_distribution(df, emotion_counts, sentiment_counts):
# ... (Fungsi ini tidak berubah)
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    emotion_counts.plot(kind='bar', ax=axes[0,0], color='skyblue', alpha=0.8)
    axes[0,0].set_title('Distribusi Emosi', fontsize=14, fontweight='bold')
    axes[0,0].set_xlabel('Emosi')
    axes[0,0].set_ylabel('Jumlah')
    axes[0,0].tick_params(axis='x', rotation=45)
    for i, v in enumerate(emotion_counts.values):
        axes[0,0].text(i, v + 20, str(v), ha='center', va='bottom')

    colors = ['lightcoral', 'lightgreen']
    axes[0,1].pie(sentiment_counts.values, labels=sentiment_counts.index,
                  autopct='%1.1f%%', colors=colors, startangle=90)
    axes[0,1].set_title('Distribusi Sentimen', fontsize=14, fontweight='bold')

    axes[1,0].hist(df['review_length'], bins=50, alpha=0.7, color='orange', edgecolor='black')
    axes[1,0].axvline(df['review_length'].mean(), color='red', linestyle='--',
                      label=f'Mean: {df["review_length"].mean():.0f}')
    axes[1,0].axvline(df['review_length'].median(), color='green', linestyle='--',
                      label=f'Median: {df["review_length"].median():.0f}')
    axes[1,0].set_title('Distribusi Panjang Review', fontsize=14, fontweight='bold')
    axes[1,0].set_xlabel('Panjang Karakter')
    axes[1,0].set_ylabel('Frekuensi')
    axes[1,0].legend()

    df.boxplot(column='review_length', by='Emotion', ax=axes[1,1])
    axes[1,1].set_title('Panjang Review per Emosi', fontsize=14, fontweight='bold')
    axes[1,1].set_xlabel('Emosi')
    axes[1,1].set_ylabel('Panjang Karakter')
    axes[1,1].tick_params(axis='x', rotation=45)

    plt.suptitle('')
    plt.tight_layout()
    plt.savefig(f'{Config.RESULTS_PATH}/data_distribution.png', dpi=300, bbox_inches='tight')
    plt.show()

def visualize_category_comparison(df):
    # Gabungkan semua kategori Fashion menjadi satu untuk perbandingan
    df['Category_Group'] = df['Category'].replace({
        "Women's Fashion": "Fashion",
        "Men's Fashion": "Fashion",
        "Muslim Fashion": "Fashion"
    })

    # Ambil 5 kategori terbesar untuk visualisasi perbandingan
    top_5_categories = df['Category_Group'].value_counts().nlargest(5).index.tolist()
    comparison_df = df[df['Category_Group'].isin(top_5_categories)]

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    fig.suptitle('Perbandingan Antar Kategori Utama (Top 5)', fontsize=16, fontweight='bold')

    # Grafik 1: Distribusi Emosi per Kategori
    sns.countplot(data=comparison_df, x='Category_Group', hue='Emotion', ax=axes[0], order=top_5_categories)
    axes[0].set_title('Distribusi Emosi per Kategori', fontsize=14)
    axes[0].set_xlabel('Kategori Produk', fontsize=12)
    axes[0].set_ylabel('Jumlah Ulasan', fontsize=12)
    axes[0].legend(title='Emosi')
    axes[0].tick_params(axis='x', rotation=45)

    # Grafik 2: Distribusi Panjang Review per Kategori
    sns.boxplot(data=comparison_df, x='Category_Group', y='review_length', ax=axes[1], order=top_5_categories)
    axes[1].set_title('Distribusi Panjang Review per Kategori', fontsize=14)
    axes[1].set_xlabel('Kategori Produk', fontsize=12)
    axes[1].set_ylabel('Panjang Review (Karakter)', fontsize=12)
    axes[1].set_ylim(0, comparison_df['review_length'].quantile(0.95)) # Fokus pada 95% data agar lebih jelas
    axes[1].tick_params(axis='x', rotation=45)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(f'{Config.RESULTS_PATH}/category_comparison_all.png', dpi=300, bbox_inches='tight')
    plt.show()

def show_emotion_examples(df):
# ... (Fungsi ini tidak berubah)
    print("\nContoh Teks untuk Setiap Emosi:")
    print("="*60)
    for emotion in df['Emotion'].unique():
        samples = df[df['Emotion'] == emotion]['Customer Review'].head(2)
        print(f"\n{emotion.upper()}:")
        for i, sample in enumerate(samples, 1):
            print(f"  {i}. \"{sample}\"")

def load_and_explore_data():
    print("="*60)
    print("LOADING, FILTERING, DAN EKSPLORASI DATA")
    print("="*60)
    df = pd.read_csv(Config.DATA_PATH, delimiter=';')
    print(f"Jumlah data awal (sebelum filter): {len(df):,}")

    # 1. Definisikan kategori yang akan digunakan
    # TIDAK DILAKUKAN: Menggunakan seluruh data

    # 2. Filter DataFrame berdasarkan kategori
    df_filtered = df.copy() # Hapus filter, gunakan salinan penuh

    print(f"\nJumlah data setelah filter: {len(df_filtered):,}")
    print(f"Kolom: {list(df_filtered.columns)}")
    print("\nInfo Dataset (setelah filter):")
    df_filtered.info()

    print("\nMissing values (setelah filter):")
    print(df_filtered.isnull().sum())

    print("\nDistribusi Kategori (setelah filter):")
    category_counts = df_filtered['Category'].value_counts()
    # Tampilkan hanya 10 kategori teratas
    print("Top 10 Kategori:")
    top_categories = category_counts.head(10)
    for category, count in top_categories.items():
        percentage = (count / len(df_filtered)) * 100
        print(f"  {category}: {count:,} ({percentage:.1f}%)")
    print(f"  ... dan {len(category_counts) - 10} kategori lainnya.")

    emotion_counts = df_filtered['Emotion'].value_counts()
    print("\nDistribusi Emosi (setelah filter):")
    for emotion, count in emotion_counts.items():
        percentage = (count / len(df_filtered)) * 100
        print(f"  {emotion}: {count:,} ({percentage:.1f}%)")

    sentiment_counts = df_filtered['Sentiment'].value_counts()
    print("\nDistribusi Sentimen (setelah filter):")
    for sentiment, count in sentiment_counts.items():
        percentage = (count / len(df_filtered)) * 100
        print(f"  {sentiment}: {count:,} ({percentage:.1f}%)")

    df_filtered['review_length'] = df_filtered['Customer Review'].str.len()
    print("\nStatistik Panjang Review (setelah filter):")
    print(f"  Rata-rata: {df_filtered['review_length'].mean():.1f} karakter")
    print(f"  Median: {df_filtered['review_length'].median():.1f} karakter")
    print(f"  Min: {df_filtered['review_length'].min()} karakter")
    print(f"  Max: {df_filtered['review_length'].max()} karakter")
    print(f"  95th percentile: {df_filtered['review_length'].quantile(0.95):.1f} karakter")

    # Panggil visualisasi dengan data yang sudah difilter
    visualize_data_distribution(df_filtered, emotion_counts, sentiment_counts)

    # Panggil visualisasi baru untuk perbandingan kategori
    print("\n" + "="*60)
    print("VISUALISASI PERBANDINGAN ANTAR KATEGORI UTAMA (Top 5)")
    print("="*60)
    visualize_category_comparison(df_filtered)

    show_emotion_examples(df_filtered)

    return df_filtered

df = load_and_explore_data()

In [ ]:
# Cell 4: Preprocessing Pipeline


def normalize_repeated_chars(text):
    text = re.sub(r'([a-zA-Z])\1{2,}', r'\1', text)
    return text

def load_slang_dict_from_url(url):
    slang_dict = {}
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        try:
            data = response.json()
            slang_dict = data
        except json.JSONDecodeError:
            # Jika bukan JSON, proses sebagai teks baris per baris
            lines = response.text.split('\n')
            for line in lines:
                # Lewati baris yang tidak valid
                if ':' not in line:
                    continue

                slang, formal = line.split(':', 1)
                # Bersihkan spasi dan tanda kutip dari key dan value
                clean_slang = slang.strip().strip('"').strip()
                clean_formal = formal.strip().strip('"').strip()

                slang_dict[clean_slang] = clean_formal

        print(f"slang berhasil dimuat dengan {len(slang_dict)} entri.")
    except requests.exceptions.RequestException as e:
        print(f"Error saat mengunduh kamus slang: {e}")
    return slang_dict

print("="*60)
print("Mempersiapkan Tools Preprocessing (Kamus Slang)")
print("="*60)
slang_url = "https://raw.githubusercontent.com/louisowen6/NLP_bahasa_resources/master/combined_slang_words.txt"
slang_dict = load_slang_dict_from_url(slang_url)
print("\nTools preprocessing siap digunakan.")


def preprocess_bert_friendly(text):
    if pd.isna(text) or text == "":
        return ""
    text = str(text)

    # Tahap 1: Case Folding
    text = text.lower()
    text = normalize_repeated_chars(text)
    # Tahap 2: Basic Cleaning
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    # Tahap 3: Normalisasi Slang (Setelah cleaning)
    words = text.split()
    normalized_words = [slang_dict.get(word, word) for word in words]
    text = ' '.join(normalized_words)

    if len(text) < 3:
        return ""

    return text

def preprocess_data(df):
    print("="*60)
    print("PREPROCESSING DATA DENGAN PIPELINE BERT-FRIENDLY (FINAL)")
    print("="*60)
    print(f"Data awal: {len(df):,}")

    print("Menerapkan pipeline (case folding, cleaning, normalisasi slang)...")
    df['processed_review'] = df['Customer Review'].apply(preprocess_bert_friendly)
    print("✓ Pipeline preprocessing selesai.")

    df_clean = df.dropna(subset=['processed_review', 'Emotion']).copy()
    df_clean = df_clean[df_clean['processed_review'].str.len() > 0]
    print(f"Data setelah preprocessing & cleaning: {len(df_clean):,}")

    before_dedup = len(df_clean)
    df_clean = df_clean.drop_duplicates(subset=['processed_review'])
    print(f"Setelah remove duplicates: {len(df_clean):,} (removed: {before_dedup - len(df_clean)})")

    label_encoder = LabelEncoder()
    df_clean['emotion_label'] = label_encoder.fit_transform(df_clean['Emotion'])

    print("\nMapping Label Emosi:")
    label_mapping = {}
    for i, emotion in enumerate(label_encoder.classes_):
        count = sum(df_clean['emotion_label'] == i)
        percentage = (count / len(df_clean)) * 100
        print(f"  {emotion}: {i} ({count:,} samples, {percentage:.1f}%)")
        label_mapping[emotion] = i
    with open(f'{Config.RESULTS_PATH}/label_mapping.json', 'w') as f:
        json.dump(label_mapping, f, indent=2)

    show_preprocessing_examples(df_clean)
    return df_clean, label_encoder
def show_preprocessing_examples(df_clean):
    print("\nContoh Hasil Preprocessing Pipeline (Final):")
    print("-"*80)
    sample_size = min(10, len(df_clean))
    if sample_size > 0:
        samples = df_clean.sample(n=sample_size, random_state=42)
        for i, (_, row) in enumerate(samples.iterrows(), 1):
            original = row['Customer Review']
            processed = row['processed_review']
            emotion = row['Emotion']
            print(f"\n{i}. Emosi: {emotion}")
            print(f"   Original:  \"{original}\"")
            print(f"   Processed: \"{processed}\"")
    else:
        print("Tidak ada data yang tersisa untuk ditampilkan setelah preprocessing.")

df_clean, label_encoder = preprocess_data(df)

In [ ]:
# Cell 5: Tokenizer Setup and Analysis
def setup_tokenizer_and_analyze(df_clean):
    print("="*60)
    print("SETUP TOKENIZER DAN ANALISIS")
    print("="*60)
    print(f"Loading tokenizer: {Config.MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)
    print(f"✓ Tokenizer loaded successfully!")
    print(f"  Vocab size: {tokenizer.vocab_size:,}")
    print(f"  Model max length: {tokenizer.model_max_length}")
    analyze_tokenization(tokenizer, df_clean)
    demonstrate_tokenization(tokenizer, df_clean)
    return tokenizer

def analyze_tokenization(tokenizer, df_clean):
    print("\n=== ANALISIS TOKENISASI ===")
    sample_size = min(1000, len(df_clean))
    sample_texts = df_clean['processed_review'].sample(n=sample_size, random_state=42).tolist()
    token_lengths = []
    word_counts = []
    print(f"Menganalisis {sample_size} samples...")
    for text in sample_texts:
        tokens = tokenizer.tokenize(text)
        token_lengths.append(len(tokens))
        words = text.split()
        word_counts.append(len(words))
    print("\nStatistik Token Length:")
    print(f"  Mean: {np.mean(token_lengths):.1f}")
    print(f"  Median: {np.median(token_lengths):.1f}")
    print(f"  Std: {np.std(token_lengths):.1f}")
    print(f"  Min: {np.min(token_lengths)}")
    print(f"  Max: {np.max(token_lengths)}")
    print(f"  95th percentile: {np.percentile(token_lengths, 95):.1f}")
    print(f"  99th percentile: {np.percentile(token_lengths, 99):.1f}")
    for max_len in [64, 128, 256, 512]:
        truncated_pct = (np.array(token_lengths) > max_len).mean() * 100
        print(f"  Data truncated at {max_len}: {truncated_pct:.1f}%")
    print("\nStatistik Word Count:")
    print(f"  Mean: {np.mean(word_counts):.1f}")
    print(f"  Ratio tokens/words: {np.mean(token_lengths)/np.mean(word_counts):.2f}")
    plot_token_distribution(token_lengths)
    return token_lengths

def plot_token_distribution(token_lengths):
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    axes[0].hist(token_lengths, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
    axes[0].axvline(Config.MAX_LENGTH, color='red', linestyle='--',
                    label=f'Max Length ({Config.MAX_LENGTH})')
    axes[0].axvline(np.mean(token_lengths), color='green', linestyle='--',
                    label=f'Mean ({np.mean(token_lengths):.1f})')
    axes[0].set_xlabel('Token Length')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribusi Panjang Token')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[1].boxplot(token_lengths)
    axes[1].set_ylabel('Token Length')
    axes[1].set_title('Box Plot Panjang Token')
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{Config.RESULTS_PATH}/token_distribution.png', dpi=300, bbox_inches='tight')
    plt.show()

def demonstrate_tokenization(tokenizer, df_clean):
    print("\n=== DEMONSTRASI TOKENISASI ===")
    sample_texts = [
        df_clean['processed_review'].iloc[0],
        df_clean[df_clean['processed_review'].str.len() > 100]['processed_review'].iloc[0],
        df_clean[df_clean['Emotion'] == 'Anger']['processed_review'].iloc[0]
    ]
    for i, text in enumerate(sample_texts, 1):
        print(f"\n--- Contoh {i} ---")
        print(f"Text: \"{text}\"")
        print(f"Length: {len(text)} characters")
        tokens = tokenizer.tokenize(text)
        print(f"Tokens ({len(tokens)}): {tokens}")
        token_ids = tokenizer.convert_tokens_to_ids(tokens)
        print(f"Token IDs: {token_ids}")
        encoding = tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=20,
            return_tensors='pt'
        )
        print(f"Full encoding (max_len=20):")
        print(f"  Input IDs: {encoding['input_ids'][0].tolist()}")
        print(f"  Attention Mask: {encoding['attention_mask'][0].tolist()}")
        decoded = tokenizer.decode(encoding['input_ids'][0], skip_special_tokens=True)
        print(f"  Decoded: \"{decoded}\"")
        print(f"Special tokens:")
        print(f"  [CLS]: {tokenizer.cls_token} (ID: {tokenizer.cls_token_id})")
        print(f"  [SEP]: {tokenizer.sep_token} (ID: {tokenizer.sep_token_id})")
        print(f"  [PAD]: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")
        print(f"  [UNK]: {tokenizer.unk_token} (ID: {tokenizer.unk_token_id})")

tokenizer = setup_tokenizer_and_analyze(df_clean)


In [ ]:
# Cell 6: Custom Dataset Class
class EmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        assert len(texts) == len(labels)
        print(f"Dataset created with {len(texts)} samples")

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt',
            add_special_tokens=True
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }


In [ ]:
# Cell 7: Data Preparation
def print_label_distribution(y, label_encoder, dataset_name):
    unique, counts = np.unique(y, return_counts=True)
    print(f"\n{dataset_name} distribution:")
    for label_idx, count in zip(unique, counts):
        emotion = label_encoder.classes_[label_idx]
        percentage = (count / len(y)) * 100
        print(f"  {emotion}: {count:,} ({percentage:.1f}%)")

def prepare_datasets(df_clean, tokenizer, label_encoder):
    print("="*60)
    print("PREPARE DATASETS")
    print("="*60)
    X = df_clean['processed_review'].values
    y = df_clean['emotion_label'].values
    print(f"Total data: {len(X):,}")
    print_label_distribution(y, label_encoder, "Original dataset")
    # Split 1: 80% (Train+Val) dan 20% (Test)
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y,
        test_size=Config.TEST_SIZE,
        random_state=Config.RANDOM_STATE,
        stratify=y
    )

    # Split 2: 80% (Train+Val) -> 72% (Train) dan 8% (Val)
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full,
        test_size=Config.VAL_SIZE,
        random_state=Config.RANDOM_STATE,
        stratify=y_train_full
    )
    print(f"\nData split results:")
    print(f"  Training: {len(X_train):,}")
    print(f"  Validation: {len(X_val):,}")
    print(f"  Testing: {len(X_test):,}")
    print_label_distribution(y_train, label_encoder, "Training set")
    print_label_distribution(y_val, label_encoder, "Validation set")
    print_label_distribution(y_test, label_encoder, "Test set")
    train_dataset = EmotionDataset(X_train, y_train, tokenizer, Config.MAX_LENGTH)
    val_dataset = EmotionDataset(X_val, y_val, tokenizer, Config.MAX_LENGTH)
    test_dataset = EmotionDataset(X_test, y_test, tokenizer, Config.MAX_LENGTH)
    return train_dataset, val_dataset, test_dataset, (X_test, y_test), y_train_full

train_dataset, val_dataset, test_dataset, (X_test, y_test), y_train_full = prepare_datasets(df_clean, tokenizer, label_encoder)


In [ ]:
# Cell 8: Model Setup


def setup_model(num_labels, device):
    print("="*60)
    print("SETUP MODEL")
    print("="*60)
    print(f"Loading model: {Config.MODEL_NAME}")
    print(f"Number of labels: {num_labels}")
    model = AutoModelForSequenceClassification.from_pretrained(
        Config.MODEL_NAME,
        num_labels=num_labels,
        output_attentions=False,
        output_hidden_states=False,
    )
    model.to(device)
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"✓ Model loaded successfully!")
    print(f"  Total parameters: {total_params:,}")
    print(f"  Trainable parameters: {trainable_params:,}")
    print(f"  Model size: ~{total_params * 4 / 1e6:.1f} MB")
    print(f"\nModel architecture:")
    print(f"  Base model: BERT")
    print(f"  Hidden size: {model.config.hidden_size}")
    print(f"  Number of layers: {model.config.num_hidden_layers}")
    print(f"  Number of attention heads: {model.config.num_attention_heads}")
    print(f"  Vocab size: {model.config.vocab_size}")
    print(f"  Max position embeddings: {model.config.max_position_embeddings}")
    return model

def model_init(trial):
    return AutoModelForSequenceClassification.from_pretrained(
        Config.MODEL_NAME,
        num_labels=num_labels
    )
num_labels = len(label_encoder.classes_)
model = setup_model(num_labels, device)


In [ ]:
# Cell 8.5: Custom Trainer for Class Weights
class CustomTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)

        if class_weights is not None:
            self.class_weights = class_weights.to(self.args.device)
        else:
            self.class_weights = None

    # Menambahkan **kwargs untuk menerima argumen ekstra seperti num_items_in_batch
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels") # Ambil dan hapus labels dari inputs

        # Forward pass
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Hitung loss dengan class weights
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

print(" CustomTrainer untuk class weights berhasil didefinisikan.")

In [ ]:
# Cell 9: Menjalankan Hyperparameter Search (Grid Search)
from optuna.samplers import GridSampler
def get_training_arguments():
    return TrainingArguments(
        output_dir=Config.LOGS_PATH,
        num_train_epochs=Config.NUM_EPOCHS, # Ini akan di-override oleh Grid Search
        per_device_train_batch_size=Config.BATCH_SIZE, # Ini akan di-override oleh Grid Search
        per_device_eval_batch_size=Config.EVAL_BATCH_SIZE, # Ini akan di-override oleh Grid Search
        learning_rate=Config.LEARNING_RATE, # Ini akan di-override oleh Grid Search
        warmup_ratio=Config.WARMUP_RATIO,
        weight_decay=Config.WEIGHT_DECAY,
        eval_strategy="epoch",
        save_strategy="no",
        logging_strategy="epoch",
        load_best_model_at_end=False,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        disable_tqdm=False,
        report_to="none"
    )

def run_hyperparameter_search(train_dataset, val_dataset, y_train_full, label_encoder, device):
    print("="*60)
    print("MENJALANKAN HYPERPARAMETER SEARCH (GRID SEARCH)")
    print("="*60)

    training_args = get_training_arguments()

    # Hitung class weights
    class_weights = compute_class_weight('balanced', classes=np.unique(y_train_full), y=y_train_full)
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

    # Inisialisasi CustomTrainer dengan model_init
    trainer = CustomTrainer(
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        model_init=model_init,
        class_weights=class_weights_tensor
    )

    def my_hp_space(trial):
      return {
          "learning_rate": trial.suggest_categorical("learning_rate", [4e-5, 3e-5, 2e-5]),
          "num_train_epochs": trial.suggest_categorical("num_train_epochs", [5, 7, 10]),
          "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [16, 32]),
          "per_device_eval_batch_size": trial.suggest_categorical("per_device_eval_batch_size", [16, 32]),
      }

      # Definisikan search space terpisah untuk GridSampler
    search_space_for_sampler = {
        "learning_rate": [4e-5, 3e-5, 2e-5],
        "num_train_epochs": [5, 7, 10],
        "per_device_train_batch_size": [16, 32],
        "per_device_eval_batch_size": [16, 32],
    }

    print("🚀 Memulai pencarian hyperparameter dengan GridSampler...")
    best_trial = trainer.hyperparameter_search(
        direction="minimize",
        backend="optuna",
        hp_space=my_hp_space,
        sampler=GridSampler(search_space_for_sampler), # GridSampler masih butuh ini
        n_trials=36
    )

    print("\n✅ Pencarian Selesai!")
    print("Hyperparameter terbaik yang ditemukan:")
    print(best_trial.hyperparameters)

    return best_trial.hyperparameters


best_params = run_hyperparameter_search(train_dataset, val_dataset, y_train_full, label_encoder, device)

2f0a1b937bb4a4bf968518c810a923321ebfb39b

In [ ]:
# Cell 9.5: Melatih Model Final dengan Hyperparameter Terbaik

print("\n" + "="*60)
print("MELATIH MODEL FINAL DENGAN HYPERPARAMETER TERBAIK")
print("="*60)

# Buat TrainingArguments baru dengan parameter terbaik dari hasil pencarian
final_training_args = TrainingArguments(
    output_dir=f"{Config.MODEL_SAVE_PATH}_best",
    learning_rate=best_params['learning_rate'],
    num_train_epochs=best_params['num_train_epochs'],
    per_device_train_batch_size=best_params['per_device_train_batch_size'],
    per_device_eval_batch_size=best_params['per_device_eval_batch_size'],
    warmup_ratio=Config.WARMUP_RATIO,
    weight_decay=Config.WEIGHT_DECAY,
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none"
)


# Hitung lagi class weights untuk trainer final
class_weights_final = compute_class_weight('balanced', classes=np.unique(y_train_full), y=y_train_full)
class_weights_tensor_final = torch.tensor(class_weights_final, dtype=torch.float).to(device)

final_trainer = CustomTrainer(
    model=model_init(None),
    args=final_training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    class_weights=class_weights_tensor_final
)

# Latih model final
final_trainer.train()
tokenizer.save_pretrained(final_training_args.output_dir)
print("\nModel terbaik (Skenario 1.1) siap untuk dievaluasi.")


In [ ]:
# Cell 10: Evaluation
def evaluate_model(trainer, test_dataset, X_test, y_test, label_encoder):
    print("="*60)
    print("MODEL EVALUATION")
    print("="*60)
    print("Making predictions on test set...")
    predictions = trainer.predict(test_dataset)
    y_pred = np.argmax(predictions.predictions, axis=1)
    y_prob = torch.softmax(torch.tensor(predictions.predictions), dim=1).numpy()
    y_true = y_test
    accuracy = accuracy_score(y_true, y_pred)
    print(f"\nTest Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    target_names = label_encoder.classes_
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=target_names))
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.show()

    # Buat DataFrame untuk analisis kesalahan
    df_analysis = pd.DataFrame({
        'text': X_test,
        'true_label': label_encoder.inverse_transform(y_true),
        'predicted_label': label_encoder.inverse_transform(y_pred)
    })

    # Filter hanya untuk prediksi yang salah
    df_errors = df_analysis[df_analysis['true_label'] != df_analysis['predicted_label']]

    # Simpan ke file CSV untuk dianalisis
    error_report_path = f'{Config.RESULTS_PATH}/error_analysis.csv'
    df_errors.to_csv(error_report_path, index=False)

    print(f"\nAnalisis kesalahan disimpan di: {error_report_path}")
    print("Contoh kesalahan prediksi:")
    print(df_errors.head(10))
    return accuracy

accuracy = evaluate_model(final_trainer, test_dataset, X_test, y_test, label_encoder)



In [ ]:
# Cell 11: Prediction Function
def predict_emotion(text, model, tokenizer, label_encoder, device, max_length=128):
    processed_text = preprocess_bert_friendly(text)
    if len(processed_text) == 0:
        return {
            'emotion': 'Unknown',
            'confidence': 0.0,
            'all_probabilities': {emotion: 0.0 for emotion in label_encoder.classes_}
        }
    encoding = tokenizer(
        processed_text,
        truncation=True,
        padding='max_length',
        max_length=max_length,
        return_tensors='pt'
    )
    encoding = {k: v.to(device) for k, v in encoding.items()}
    model.eval()
    with torch.no_grad():
        outputs = model(**encoding)
        predictions = F.softmax(outputs.logits, dim=-1)
        predicted_class = torch.argmax(predictions, dim=-1).item()
        confidence = predictions[0][predicted_class].item()
    emotion = label_encoder.classes_[predicted_class]
    all_probs = {
        label_encoder.classes_[i]: predictions[0][i].item()
        for i in range(len(label_encoder.classes_))
    }
    return {
        'emotion': emotion,
        'confidence': confidence,
        'all_probabilities': all_probs,
        'processed_text': processed_text
    }

# Example usage
test_text = "Jelek banget dah, wah kacaw"
result = predict_emotion(test_text, model, tokenizer, label_encoder, device)
print(f"Text: \"{test_text}\"")
print(f"Predicted Emotion: {result['emotion']} (confidence: {result['confidence']:.3f})")
